# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We'll enumerate all record sets available in the schema. For each record set, we'll print its `@id`, name, and list field IDs and names.

In [ ]:
# List out all available record sets and their fields
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RECORD SET @id: {rs.id}")
    print(f"  Name: {rs.name if hasattr(rs, 'name') else '(no name)'}")
    print(f"  Fields:")
    for fld in getattr(rs, 'fields', []):
        print(f"    FIELD @id: {fld.id} | name: {fld.name if hasattr(fld,'name') else '(no name)'}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We'll load each record set referenced by `@id` using mlcroissant, and preview its columns.

In [ ]:
# Extract data from each record set
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record set {record_set_id} loaded: shape={df.shape}")
if record_set_ids:
    print(f"Columns in first record set ({record_set_ids[0]}):\n", dataframes[record_set_ids[0]].columns.tolist())
    display(dataframes[record_set_ids[0]].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filtering records, normalizing numeric fields, and grouping by categorical fields.

We'll select a record set and pick one numeric and one group field by their `@id` (chosen from inspection above).

In [ ]:
# For EDA, pick record set and field IDs found above
if record_set_ids:
    record_set_id = record_set_ids[0]  # pick the first one for demo
    df = dataframes[record_set_id]
    print(f"Working with record set: {record_set_id}, columns: {df.columns.tolist()}")
    
    # Try picking a numeric field (look for likely numerics from column names)
    numeric_fields = [c for c in df.columns if df[c].dtype in ['int64', 'float64'] or c.lower().startswith(('count','abundance','value','mw','mass','percent','peptide','coverage'))]
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())
        # Normalize numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try grouping by a group field (categorical)
        group_fields = [c for c in df.columns if c != numeric_field and (df[c].dtype == 'object' or df[c].dtype.name == 'category')]
        if group_fields:
            group_field = group_fields[0]
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"\nGrouped data by {group_field}:")
            print(grouped_df.head())
    else:
        print("No numeric fields found for analysis.")
else:
    print("No record sets found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot a histogram for a numeric field and a boxplot grouped by a categorical field, if present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    numeric_fields = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if numeric_fields:
        numeric_field = numeric_fields[0]
        plt.figure(figsize=(8, 4))
        sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
        plt.title(f'Histogram of {numeric_field}')
        plt.xlabel(numeric_field)
        plt.ylabel('Frequency')
        plt.show()

        # Boxplot by group if a group field exists
        group_fields = [c for c in df.columns if c != numeric_field and (df[c].dtype == 'object' or df[c].dtype.name == 'category')]
        if group_fields:
            group_field = group_fields[0]
            plt.figure(figsize=(10,5))
            sns.boxplot(data=df, x=group_field, y=numeric_field)
            plt.title(f'Boxplot of {numeric_field} by {group_field}')
            plt.xticks(rotation=45)
            plt.show()
    else:
        print('No numeric field to visualize.')
else:
    print('No record sets to visualize.')

## 6. Conclusion
In this notebook, we loaded mass spectrometry data from a Croissant-structured FAIR dataset using `mlcroissant`, previewed record set structures, explored fields by their `@id`, performed simple filtering and normalization, and visualized key numeric attributes. This workflow can be adapted to other Croissant-structured datasets for reproducible, FAIR data analysis.